**Status: canonical pipeline.** Lap-level podium/win probability models (`pre_model`, `live_model`), saved to `models/pre_model.json` / `models/live_model.json` and loaded by `notebooks/phase_4.ipynb`.

A separate, more elaborate stint-level exploration lives in `notebooks/phase_3_strategy.ipynb` (full pit-strategy reconstruction: compound + pit lap + win probability). Its artifacts are wired into `notebooks/phase_4.ipynb` and the Databricks App independently of `pre_model`/`live_model` — treat it as a parallel research track, not a step in this pipeline; the two aren't fused into one prediction yet.

In [ ]:
# Spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ML utilities
import numpy as np
import pandas as pd

# XGBoost — XGBRanker instead of XGBClassifier (see Plan A: podium/win
# prediction is reframed as learning-to-rank, grouped per race/lap-snapshot
# via qid, rather than independent per-driver binary classification).
from xgboost import XGBRanker

# MLflow — experiment tracking + Unity Catalog model registry, replacing
# the old approach of only saving pre_model.json/live_model.json locally.
import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

# Sklearn helpers. ndcg_score/brier_score_loss/log_loss and
# IsotonicRegression/calibration_curve support Plan F's formalized ranking
# metrics and reinstated (ranker-appropriate) calibration further down.
from sklearn.metrics import roc_auc_score, ndcg_score, brier_score_loss, log_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

# scipy — Kendall's tau, the whole-field ranking-correctness metric added
# in Plan F alongside NDCG@3.
from scipy.stats import kendalltau

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("F1-Strategy-Engine")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

In [0]:
MASTER_PATH = "workspace.default.f1_cleaned_lap_dataset"

df = spark.table(MASTER_PATH)

In [0]:
display(df)

In [0]:
print(df.columns)

In [0]:
stint_window = (
    Window
    .partitionBy("Driver", "Year", "Circuit", "Stint")
    .orderBy("LapNumber")
)

df = df.withColumn(
    "lap_delta",
    F.col("LapTime") - F.lag("LapTime").over(stint_window)
)

df = df.withColumn(
    "lap_delta",
    F.when((F.col("lap_delta") < -5) | (F.col("lap_delta") > 5), None)
     .otherwise(F.col("lap_delta"))
)

In [ ]:
# 1. Race normalized lap pace
race_avg = df.groupBy("Year","Circuit","LapNumber")              .agg(F.avg("LapTime").alias("race_avg_laptime"))

df = df.join(race_avg, ["Year","Circuit","LapNumber"])

df = df.withColumn(
    "relative_laptime",
    F.col("LapTime") - F.col("race_avg_laptime")
)

# 2. Sector normalization
sector_avg = df.groupBy("Year","Circuit").agg(
    F.avg("Sector1Time").alias("sector1_avg"),
    F.avg("Sector2Time").alias("sector2_avg"),
    F.avg("Sector3Time").alias("sector3_avg")
)

df = df.join(sector_avg, ["Year","Circuit"])

df = df.withColumn("sector1_rel", F.col("Sector1Time") - F.col("sector1_avg"))
df = df.withColumn("sector2_rel", F.col("Sector2Time") - F.col("sector2_avg"))
df = df.withColumn("sector3_rel", F.col("Sector3Time") - F.col("sector3_avg"))

# 3. Overtaking ability
# This stays as a race-relative proxy and is reused in the driver profile below.
df = df.withColumn(
    "position_gain",
    F.col("QualiPosition") - F.col("FinalPosition")
)

# 4. Tyre management
# Grouped by (Driver, Year), not Driver alone — see the leakage note in the
# driver-profile cell below for why the Year dimension matters here.
stint_life = df.groupBy("Driver","Year","Stint")                .agg(F.max("TyreLife").alias("stint_life"))

tyre_management = stint_life.groupBy("Driver","Year")                             .agg(F.avg("stint_life").alias("tyre_management"))

# 5. Driver consistency
consistency = df.groupBy("Driver","Year")                 .agg(F.stddev("LapTime").alias("lap_consistency"))


In [ ]:
# Tyre degradation rate: slope of relative pace (relative_laptime, which
# already has the race's shared per-lap pace trend — fuel burn-off, track
# evolution — removed) against TyreLife within each stint. Computed via the
# closed-form least-squares slope so it's a handful of Spark window
# aggregates instead of a per-group Python/pandas regression. A positive
# slope means the driver is losing time as the tyre ages (degrading);
# requires at least 3 laps in the stint to avoid fitting a slope through
# 1-2 noisy points.
degradation_window = Window.partitionBy("Driver", "Year", "Circuit", "Stint")

df = (
    df
    .withColumn("_deg_n", F.count(F.col("relative_laptime")).over(degradation_window))
    .withColumn("_deg_sum_x", F.sum("TyreLife").over(degradation_window))
    .withColumn("_deg_sum_y", F.sum("relative_laptime").over(degradation_window))
    .withColumn("_deg_sum_xy", F.sum(F.col("TyreLife") * F.col("relative_laptime")).over(degradation_window))
    .withColumn("_deg_sum_xx", F.sum(F.col("TyreLife") * F.col("TyreLife")).over(degradation_window))
)

_deg_denom = F.col("_deg_n") * F.col("_deg_sum_xx") - F.col("_deg_sum_x") * F.col("_deg_sum_x")

df = df.withColumn(
    "tyre_degradation_rate",
    F.when(
        (F.col("_deg_n") >= 3) & (_deg_denom != 0),
        (F.col("_deg_n") * F.col("_deg_sum_xy") - F.col("_deg_sum_x") * F.col("_deg_sum_y")) / _deg_denom
    ).otherwise(F.lit(None).cast("double"))
).drop("_deg_n", "_deg_sum_x", "_deg_sum_y", "_deg_sum_xy", "_deg_sum_xx")

In [ ]:
# 6. Driver profile
# Build a season-normalized, car-light profile using race outcomes relative to field size
# and teammate comparisons instead of raw constructor-dominant results.
status_agg = (
    F.max(F.coalesce(F.col("Status"), F.lit(""))).alias("race_status")
    if "Status" in df.columns
    else F.first(F.lit("Finished")).alias("race_status")
)

# driver_style_profile is now grouped by (Driver, Year), not Driver alone.
# The original version collapsed it across all six seasons and joined that
# single cross-year profile onto every row regardless of year — so a 2018
# row's features were quietly built using 2019-2023 results too (and vice
# versa), leaking future outcomes into earlier training rows and letting
# test-year outcomes bleed into what the model saw as "training" features.
# Keeping Year here and joining on (Driver, Year) below scopes each row's
# profile to that driver's OTHER races within the same season only.
# (This still doesn't prevent a mid-season race from including LATER races
# in that same season — a real, narrower, known limitation not addressed
# in this pass.)
driver_style_profile = df.groupBy("Driver", "Year").agg(
    F.avg("relative_laptime").alias("driver_pace"),
    F.avg("sector1_rel").alias("driver_sector1_skill"),
    F.avg("sector2_rel").alias("driver_sector2_skill"),
    F.avg("sector3_rel").alias("driver_sector3_skill"),
    F.avg("position_gain").alias("driver_overtake_skill"),
    F.avg("SpeedI1").alias("driver_speedI1"),
    F.avg("SpeedI2").alias("driver_speedI2"),
    F.avg("SpeedFL").alias("driver_speedFL"),
    F.avg("SpeedST").alias("driver_speedST")
)

driver_race_results = df.groupBy("Year", "Circuit", "Driver", "TeamName").agg(
    F.max("QualiPosition").alias("quali_position"),
    F.max("FinalPosition").alias("final_position"),
    F.max("LapNumber").alias("laps_completed"),
    F.max(
        F.when(
            F.upper(F.coalesce(F.col("Compound"), F.lit(""))).isin("INTERMEDIATE", "WET"),
            1
        ).otherwise(0)
    ).alias("wet_race_flag"),
    status_agg
)

event_window = Window.partitionBy("Year", "Circuit")
team_window = Window.partitionBy("Year", "Circuit", "TeamName")
field_denominator = F.when(F.col("field_size") > 1, F.col("field_size") - 1).otherwise(F.lit(1.0))
status_text = F.lower(F.col("race_status"))
classified_finish = (
    status_text.contains("finished") |
    status_text.startswith("+") |
    status_text.contains("lap")
)

driver_race_results = (
    driver_race_results
    .withColumn("field_size", F.count("*").over(event_window))
    .withColumn("team_size", F.count("*").over(team_window))
    .withColumn("race_max_laps", F.max("laps_completed").over(event_window))
    .withColumn("finish_score", 1 - ((F.col("final_position") - 1) / field_denominator))
    .withColumn("quali_score", 1 - ((F.col("quali_position") - 1) / field_denominator))
    .withColumn("quali_vs_race_delta", F.col("finish_score") - F.col("quali_score"))
    .withColumn("positions_gained_norm", (F.col("quali_position") - F.col("final_position")) / field_denominator)
    .withColumn("win_flag", (F.col("final_position") == 1).cast("int"))
    .withColumn("podium_flag", (F.col("final_position") <= 3).cast("int"))
    .withColumn(
        "dnf_flag",
        F.when(classified_finish | (F.col("laps_completed") >= 0.9 * F.col("race_max_laps")), 0).otherwise(1)
    )
)

driver_race_results = (
    driver_race_results
    .withColumn("team_finish_total", F.sum("finish_score").over(team_window))
    .withColumn("team_quali_total", F.sum("quali_score").over(team_window))
    .withColumn("team_gain_total", F.sum("positions_gained_norm").over(team_window))
    .withColumn(
        "teammate_finish_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("finish_score") - ((F.col("team_finish_total") - F.col("finish_score")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "teammate_quali_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("quali_score") - ((F.col("team_quali_total") - F.col("quali_score")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "teammate_gain_delta",
        F.when(
            F.col("team_size") > 1,
            F.col("positions_gained_norm") - ((F.col("team_gain_total") - F.col("positions_gained_norm")) / (F.col("team_size") - 1))
        ).otherwise(F.lit(0.0))
    )
)

driver_season_profile = driver_race_results.groupBy("Driver", "Year").agg(
    F.avg("finish_score").alias("driver_avg_finish_score_season"),
    F.avg("quali_vs_race_delta").alias("driver_quali_vs_race_delta_season"),
    F.stddev("finish_score").alias("driver_finish_consistency_season"),
    F.avg("win_flag").alias("driver_win_rate_season"),
    F.avg("podium_flag").alias("driver_podium_rate_season"),
    F.avg(F.when(F.col("wet_race_flag") == 1, F.col("finish_score"))).alias("driver_wet_finish_score_season"),
    F.avg(F.when(F.col("wet_race_flag") == 0, F.col("finish_score"))).alias("driver_dry_finish_score_season"),
    F.avg("positions_gained_norm").alias("driver_overtake_ability_season"),
    F.avg("dnf_flag").alias("driver_dnf_rate_season"),
    F.avg("teammate_finish_delta").alias("driver_teammate_finish_delta_season"),
    F.avg("teammate_quali_delta").alias("driver_teammate_quali_delta_season"),
    F.avg("teammate_gain_delta").alias("driver_teammate_gain_delta_season")
)

# This used to go one step further and re-aggregate driver_season_profile
# across ALL years down to one row per Driver, discarding the Year
# dimension right before joining onto the main dataframe by Driver alone —
# the actual mechanism of the leak described above. Keeping the
# (Driver, Year) grain here (just renaming columns to drop the "_season"
# suffix) and joining on (Driver, Year) below fixes it, without changing
# any of the exposed column names downstream.
driver_skill_profile = driver_season_profile.select(
    "Driver", "Year",
    F.col("driver_avg_finish_score_season").alias("driver_avg_finish_score"),
    F.col("driver_quali_vs_race_delta_season").alias("driver_quali_vs_race_delta"),
    F.col("driver_finish_consistency_season").alias("driver_finish_consistency"),
    F.col("driver_win_rate_season").alias("driver_win_rate"),
    F.col("driver_podium_rate_season").alias("driver_podium_rate"),
    F.col("driver_wet_finish_score_season").alias("driver_wet_finish_score"),
    F.col("driver_dry_finish_score_season").alias("driver_dry_finish_score"),
    F.col("driver_overtake_ability_season").alias("driver_overtake_ability"),
    F.col("driver_dnf_rate_season").alias("driver_dnf_rate"),
    F.col("driver_teammate_finish_delta_season").alias("driver_teammate_finish_delta"),
    F.col("driver_teammate_quali_delta_season").alias("driver_teammate_quali_delta"),
    F.col("driver_teammate_gain_delta_season").alias("driver_teammate_gain_delta"),
)

driver_profile = (
    driver_style_profile
    .join(consistency, ["Driver", "Year"], "left")
    .join(tyre_management, ["Driver", "Year"], "left")
    .join(driver_skill_profile, ["Driver", "Year"], "left")
)

print(driver_profile.columns)


In [ ]:
# 7. Constructor profile
# Capture constructor strength and reliability separately so the notebook can model car/team effects
# without mixing them into the driver profile itself.
constructor_race_results = driver_race_results.groupBy("Year", "Circuit", "TeamName").agg(
    F.avg("finish_score").alias("constructor_finish_score_race"),
    F.avg("quali_score").alias("constructor_quali_score_race"),
    F.avg("win_flag").alias("constructor_win_rate_race"),
    F.avg("podium_flag").alias("constructor_podium_rate_race"),
    F.avg("dnf_flag").alias("constructor_dnf_rate_race")
)

constructor_season_profile = constructor_race_results.groupBy("TeamName", "Year").agg(
    F.avg("constructor_finish_score_race").alias("constructor_avg_finish_score_season"),
    F.avg("constructor_quali_score_race").alias("constructor_avg_quali_score_season"),
    F.avg("constructor_win_rate_race").alias("constructor_win_rate_season"),
    F.avg("constructor_podium_rate_race").alias("constructor_podium_rate_season"),
    F.avg("constructor_dnf_rate_race").alias("constructor_dnf_rate_season")
)

# Same fix as driver_skill_profile above: keep the (TeamName, Year) grain
# instead of collapsing constructor_season_profile across all six seasons
# and joining by TeamName alone. A team's car changes every season — a
# cross-year average was blending, say, a competitive 2022 Ferrari with a
# struggling 2018 Ferrari into one number applied to both years alike.
constructor_profile = constructor_season_profile.select(
    "TeamName", "Year",
    F.col("constructor_avg_finish_score_season").alias("constructor_avg_finish_score"),
    F.col("constructor_avg_quali_score_season").alias("constructor_avg_quali_score"),
    F.col("constructor_win_rate_season").alias("constructor_win_rate"),
    F.col("constructor_podium_rate_season").alias("constructor_podium_rate"),
    F.col("constructor_dnf_rate_season").alias("constructor_dnf_rate"),
    F.col("constructor_avg_finish_score_season").alias("constructor_season_strength"),
)

print(constructor_profile.columns)


In [ ]:
# 8. Merge driver and constructor profiles into main dataframe
# Joined on (Driver, Year) / (TeamName, Year) now, matching the
# season-scoped profiles built above — not on Driver/TeamName alone.
df = df.join(driver_profile, ["Driver", "Year"], "left")
df = df.join(constructor_profile, ["TeamName", "Year"], "left")

display(df)


### **Step 2: Target Construction**

In [0]:
finish_window = Window.partitionBy("Year", "Circuit", "Driver")

df = df.withColumn(
    "FinalPosition",
    F.max("Position").over(finish_window)
)

In [ ]:
df = (
    df
    .withColumn("PodiumFinish", (F.col("FinalPosition") <= 3).cast("int"))
    .withColumn("WinFinish", (F.col("FinalPosition") == 1).cast("int"))
    # Graded relevance label for the XGBRanker models below: P1 -> 20, down
    # to P20 -> 1, unclassified/no position -> 0. This gives the ranker
    # partial credit for near misses instead of the flat 0/1 PodiumFinish
    # split, while PodiumFinish/WinFinish are kept as-is for evaluation.
    .withColumn(
        "Relevance",
        F.when(
            F.col("FinalPosition").isNull(), F.lit(0)
        ).otherwise(
            F.greatest(F.lit(21) - F.col("FinalPosition"), F.lit(0))
        ).cast("int")
    )
)

In [0]:
weather_window = Window.partitionBy("Year", "Circuit")

df = (
    df
    .withColumn("AirTemp_delta", F.col("AirTemp_C") - F.avg("AirTemp_C").over(weather_window))
    .withColumn("TrackTemp_delta", F.col("TrackTemp_C") - F.avg("TrackTemp_C").over(weather_window))
    .withColumn("Humidity_delta", F.col("Humidity_pct") - F.avg("Humidity_pct").over(weather_window))
    .withColumn("WindSpeed_delta", F.col("WindSpeed_kmh") - F.avg("WindSpeed_kmh").over(weather_window))
)

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.default.f1_ml_lap_dataset")

### **Step 3: Feature Selection**

In [0]:
FEATURES_PRERACE = [
    "Driver",
    "TeamName",
    "Circuit",
    "Year",
    "QualiPosition",

    # # Driver style
    # 'driver_pace',
    # 'driver_sector1_skill',
    # 'driver_sector2_skill',
    # 'driver_sector3_skill',
    # 'driver_overtake_skill',
    # 'driver_speedI1',
    # 'driver_speedI2',
    # 'driver_speedFL',
    # 'driver_speedST',
    # 'lap_consistency',
    # 'tyre_management',

    # # Driver profile
    # 'driver_avg_finish_score',
    # 'driver_quali_vs_race_delta',
    # 'driver_finish_consistency',
    # 'driver_win_rate',
    # 'driver_podium_rate',
    # 'driver_wet_finish_score',
    # 'driver_dry_finish_score',
    # 'driver_overtake_ability',
    # 'driver_dnf_rate',
    # 'driver_teammate_finish_delta',
    # 'driver_teammate_quali_delta',
    # 'driver_teammate_gain_delta',

    # # Constructor profile
    # 'constructor_avg_finish_score',
    # 'constructor_avg_quali_score',
    # 'constructor_win_rate',
    # 'constructor_podium_rate',
    # 'constructor_dnf_rate',
    # 'constructor_season_strength',
]


In [ ]:
FEATURES_LIVE = [
    "LapNumber",
    "RacePhase",
    "Position",
    "GapToAhead",
    "DeltaToLeader",

    "Compound",
    "TyreLife",
    "Stint",
    "TrackStatus",
    "tyre_degradation_rate",

    # # Driver style
    # 'driver_pace',
    # 'driver_sector1_skill',
    # 'driver_sector2_skill',
    # 'driver_sector3_skill',
    # 'driver_overtake_skill',
    # 'driver_speedI1',
    # 'driver_speedI2',
    # 'driver_speedFL',
    # 'driver_speedST',
    # 'lap_consistency',
    # 'tyre_management',

    # # Driver profile
    # 'driver_avg_finish_score',
    # 'driver_quali_vs_race_delta',
    # 'driver_finish_consistency',
    # 'driver_win_rate',
    # 'driver_podium_rate',
    # 'driver_wet_finish_score',
    # 'driver_dry_finish_score',
    # 'driver_overtake_ability',
    # 'driver_dnf_rate',
    # 'driver_teammate_finish_delta',
    # 'driver_teammate_quali_delta',
    # 'driver_teammate_gain_delta',

    # # Constructor profile
    # 'constructor_avg_finish_score',
    # 'constructor_avg_quali_score',
    # 'constructor_win_rate',
    # 'constructor_podium_rate',
    # 'constructor_dnf_rate',
    # 'constructor_season_strength',

    # Weather (relative only)
    "AirTemp_delta",
    "TrackTemp_delta",
    "Humidity_delta",
    "WindSpeed_delta"
]


### **Step 4: Convert to Pandas**

In [ ]:
df_prerace = (
    df.select(FEATURES_PRERACE + ["PodiumFinish", "FinalPosition", "Relevance"])
      # One row per driver per race — FEATURES_PRERACE is entirely
      # race-constant (Driver/TeamName/Circuit/Year/QualiPosition don't
      # change lap to lap), but selecting straight from the lap-level `df`
      # without this would carry ~55 identical duplicate rows per
      # driver-race (one per lap). That silently over-weighted races with
      # more laps, and would have made the ranking groups below
      # nonsensical (a "race" group padded with dozens of duplicate
      # entries per driver instead of one row per competitor).
      .dropDuplicates(["Year", "Circuit", "Driver"])
)
# "Year" isn't in FEATURES_LIVE (it's not a feature the live model trains on)
# but we need it, along with Circuit, to build the year-based split and the
# per-race/per-lap ranking groups below.
df_live = df.select(FEATURES_LIVE + ["Year", "Circuit", "PodiumFinish", "FinalPosition", "Relevance"])

pdf_prerace = df_prerace.toPandas()
pdf_live = df_live.toPandas()

In [0]:
CATEGORICAL_PRERACE = ["Driver", "TeamName", "Circuit"]
CATEGORICAL_LIVE = ["Compound", "RacePhase"]

for c in CATEGORICAL_PRERACE:
    pdf_prerace[c] = pdf_prerace[c].astype("category")

for c in CATEGORICAL_LIVE:
    pdf_live[c] = pdf_live[c].astype("category")

In [ ]:
def topk_precision(scores_df, group_cols, score_col, true_col, k=3):
    """
    Ranking-appropriate accuracy check, since a raw XGBRanker score isn't a
    probability and calibration curves don't apply to it directly. For each
    group (a race, or a single lap-snapshot across the field), take the
    top-k rows by predicted score and measure what fraction actually have
    true_col == 1 — e.g. "of the 3 drivers we ranked highest, how many
    were actually on the podium".
    """
    def _precision(group):
        top = group.nlargest(k, score_col)
        return top[true_col].sum() / k

    return scores_df.groupby(group_cols).apply(_precision).mean()


# --- Plan F: formalized ranking metrics + race-level bootstrap CI --------
# topk_precision above is a coarse proxy: it only checks whether the top-k
# *set* overlaps the true podium, ignoring order within that set and
# ignoring everything outside it. These three functions plus bootstrap_ci
# are the "make it rigorous" pass — real NDCG@3 (order-sensitive, uses the
# full graded Relevance label), Kendall's tau (whole-field rank
# correctness, not just top-k), and a bootstrap band computed by
# resampling *races*, not rows, so the CI reflects the true sample size
# (tens of races per test split, not tens of thousands of rows).

def topk_precision_group(group, score_col, true_col, k=3):
    """Per-group top-k precision — the building block bootstrap_ci resamples over."""
    kk = min(k, len(group))
    if kk == 0:
        return np.nan
    top = group.nlargest(kk, score_col)
    return top[true_col].sum() / kk


def ndcg_at_k_group(group, score_col, relevance_col, k=3):
    """
    Real NDCG@k for one race/lap-snapshot group, using sklearn's ndcg_score
    against the graded Relevance label (P1=20 ... down to 0), not just the
    binary podium flag topk_precision uses. This additionally rewards
    getting the *order* of the top-k right — predicting P1 first is worth
    more than predicting P1 third — which topk_precision can't see.
    """
    if len(group) < 2:
        return np.nan
    y_true = group[relevance_col].to_numpy().reshape(1, -1)
    y_score = group[score_col].to_numpy().reshape(1, -1)
    kk = min(k, group.shape[0])
    return ndcg_score(y_true, y_score, k=kk)


def kendall_tau_group(group, score_col, relevance_col):
    """
    Kendall's tau between predicted score order and actual finishing order
    for one group — a whole-field ranking-correctness check, since a model
    can nail the podium and still scramble P10-P20 without either
    topk_precision or NDCG@3 noticing.
    """
    if len(group) < 2 or group[relevance_col].nunique() < 2:
        return np.nan
    tau, _ = kendalltau(group[score_col], group[relevance_col])
    return tau


def bootstrap_ci(values, n_boot=2000, ci=0.95, seed=42):
    """
    Percentile bootstrap CI over a *race-level* metric array (one value per
    race/group, not per row). With only ~20-30 test races per split, a
    single point estimate is mostly noise — this resamples groups with
    replacement so the CI reflects the actual number of independent races
    the metric was measured over, not the much larger row count.
    Returns (lo, mean, hi).
    """
    values = np.asarray(list(values), dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boot_means = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_boot)
    ])
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return (lo, float(values.mean()), hi)

### **Step 6: Train Pre-Race Strategy Probability Model**

In [ ]:
# Sort by race so identical-race rows are contiguous — required for
# XGBRanker's qid grouping (rows sharing a qid must not be interleaved
# with rows from another group).
pdf_prerace = pdf_prerace.sort_values(["Year", "Circuit"]).reset_index(drop=True)
pdf_prerace["qid"] = pdf_prerace.groupby(["Year", "Circuit"], sort=True).ngroup()

X_pre = pdf_prerace[FEATURES_PRERACE]
y_pre = pdf_prerace["Relevance"]
qid_pre = pdf_prerace["qid"]

# ---------# Time-based split
# ---------
TRAIN_END_YEAR = 2020
VAL_YEAR = 2021
TEST_START_YEAR = 2022

# Train
train_mask = pdf_prerace["Year"] <= TRAIN_END_YEAR

# Validation (optional but recommended for tuning)
val_mask = pdf_prerace["Year"] == VAL_YEAR

# Test (future simulation)
test_mask = pdf_prerace["Year"] >= TEST_START_YEAR

X_pre_train = X_pre.loc[train_mask].copy()
y_pre_train = y_pre.loc[train_mask].copy()
qid_pre_train = qid_pre.loc[train_mask].copy()

X_pre_val = X_pre.loc[val_mask].copy()
y_pre_val = y_pre.loc[val_mask].copy()
qid_pre_val = qid_pre.loc[val_mask].copy()

X_pre_test = X_pre.loc[test_mask].copy()
y_pre_test = y_pre.loc[test_mask].copy()
qid_pre_test = qid_pre.loc[test_mask].copy()

# Kept for a like-for-like comparison against the old classifier's AUC.
podium_pre_test = pdf_prerace.loc[test_mask, "PodiumFinish"].copy()

In [0]:
pdf_prerace.head()

In [ ]:
mlflow.set_registry_uri("databricks-uc")

pre_model_params = dict(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="rank:ndcg",
    eval_metric="ndcg",
    tree_method="hist",
    early_stopping_rounds=30,
)

with mlflow.start_run(run_name="pre_model_ranker") as run:
    mlflow.log_params(pre_model_params)

    pre_model = XGBRanker(**pre_model_params)
    # X_pre_val/y_pre_val (the 2021 season) are wired as eval_set so
    # XGBoost stops boosting once the held-out year stops improving,
    # instead of always training all 400 rounds.
    pre_model.fit(
        X_pre_train, y_pre_train, qid=qid_pre_train,
        eval_set=[(X_pre_val, y_pre_val)],
        eval_qid=[qid_pre_val],
        verbose=False,
    )

    # XGBRanker has no predict_proba — .predict() returns a raw relevance
    # score, only meaningful relative to other rows in the same qid group.
    pre_scores_test = pre_model.predict(X_pre_test)

    # AUC against the old binary PodiumFinish label, for a like-for-like
    # comparison against the previous classifier's reported number.
    # roc_auc_score accepts any real-valued score as its second argument,
    # not just a calibrated probability, so this is still a valid check.
    pre_auc = roc_auc_score(podium_pre_test, pre_scores_test)

    # Ranking-native metric: of the 3 drivers we ranked highest per race,
    # how many actually finished on the podium.
    eval_pre_test = pdf_prerace.loc[test_mask, ["Year", "Circuit", "PodiumFinish", "Relevance"]].copy()
    eval_pre_test["score"] = pre_scores_test
    pre_top3 = topk_precision(eval_pre_test, ["Year", "Circuit"], "score", "PodiumFinish", k=3)

    # Plan F: formalized ranking metrics (NDCG@3, Kendall's tau) plus
    # race-level bootstrap CIs, computed from the same per-race arrays so
    # the CI reflects the true number of independent test races (~tens),
    # not the much larger row count.
    pre_race_topk = eval_pre_test.groupby(["Year", "Circuit"]).apply(
        lambda g: topk_precision_group(g, "score", "PodiumFinish", k=3)
    )
    pre_race_ndcg = eval_pre_test.groupby(["Year", "Circuit"]).apply(
        lambda g: ndcg_at_k_group(g, "score", "Relevance", k=3)
    )
    pre_race_tau = eval_pre_test.groupby(["Year", "Circuit"]).apply(
        lambda g: kendall_tau_group(g, "score", "Relevance")
    )

    pre_ndcg3 = pre_race_ndcg.mean()
    pre_tau = pre_race_tau.mean()
    pre_top3_ci = bootstrap_ci(pre_race_topk)
    pre_ndcg3_ci = bootstrap_ci(pre_race_ndcg)
    pre_tau_ci = bootstrap_ci(pre_race_tau)

    mlflow.log_metric("test_auc_vs_podium", pre_auc)
    mlflow.log_metric("test_top3_precision", pre_top3)
    mlflow.log_metric("test_ndcg_at_3", pre_ndcg3)
    mlflow.log_metric("test_kendall_tau", pre_tau)
    mlflow.log_metric("test_top3_precision_ci_lo", pre_top3_ci[0])
    mlflow.log_metric("test_top3_precision_ci_hi", pre_top3_ci[2])
    mlflow.log_metric("test_ndcg_at_3_ci_lo", pre_ndcg3_ci[0])
    mlflow.log_metric("test_ndcg_at_3_ci_hi", pre_ndcg3_ci[2])
    mlflow.log_metric("test_kendall_tau_ci_lo", pre_tau_ci[0])
    mlflow.log_metric("test_kendall_tau_ci_hi", pre_tau_ci[2])
    mlflow.log_metric("test_n_races", int(eval_pre_test.groupby(["Year", "Circuit"]).ngroups))

    pre_model_signature = infer_signature(X_pre_train, pre_model.predict(X_pre_train))
    mlflow.xgboost.log_model(
        pre_model,
        "pre_model",
        signature=pre_model_signature,
        registered_model_name="workspace.default.f1_pre_model",
    )

print("Pre-race AUC vs PodiumFinish (unseen races):", round(pre_auc, 4))
print(f"Pre-race top-3 precision: {pre_top3:.4f}  (95% CI {pre_top3_ci[0]:.4f}-{pre_top3_ci[2]:.4f})")
print(f"Pre-race NDCG@3:          {pre_ndcg3:.4f}  (95% CI {pre_ndcg3_ci[0]:.4f}-{pre_ndcg3_ci[2]:.4f})")
print(f"Pre-race Kendall's tau:   {pre_tau:.4f}  (95% CI {pre_tau_ci[0]:.4f}-{pre_tau_ci[2]:.4f})")
print("Test races:", eval_pre_test.groupby(["Year", "Circuit"]).ngroups)

### **Step 5: Train Live Podium Probability Model**

In [ ]:
# Same year-based split as the pre-race model (TRAIN_END_YEAR / VAL_YEAR /
# TEST_START_YEAR, defined above), instead of:
#   groups = pdf_live.index; GroupShuffleSplit(...).split(pdf_live, groups=groups)
# Using each row's own index as its "group" made GroupShuffleSplit degenerate
# into a plain random split: it ignored year entirely and let laps from the
# same race land on both sides of the split. Consecutive laps are heavily
# autocorrelated (position/gap/compound barely change lap-to-lap), so that
# was inflating the reported AUC with near-duplicate train/test rows.
#
# qid groups drivers by (Year, Circuit, LapNumber) — i.e. "given everyone's
# state at this exact point in this race, rank them by podium likelihood" —
# rather than by whole race, since this model only ever sees lap-level state.
pdf_live = pdf_live.sort_values(["Year", "Circuit", "LapNumber"]).reset_index(drop=True)
pdf_live["qid"] = pdf_live.groupby(["Year", "Circuit", "LapNumber"], sort=True).ngroup()

train_mask_live = pdf_live["Year"] <= TRAIN_END_YEAR
val_mask_live = pdf_live["Year"] == VAL_YEAR
test_mask_live = pdf_live["Year"] >= TEST_START_YEAR

X_train = pdf_live.loc[train_mask_live, FEATURES_LIVE].copy()
y_train = pdf_live.loc[train_mask_live, "Relevance"].copy()
qid_train = pdf_live.loc[train_mask_live, "qid"].copy()

X_val_live = pdf_live.loc[val_mask_live, FEATURES_LIVE].copy()
y_val_live = pdf_live.loc[val_mask_live, "Relevance"].copy()
qid_val_live = pdf_live.loc[val_mask_live, "qid"].copy()

X_test = pdf_live.loc[test_mask_live, FEATURES_LIVE].copy()
y_test = pdf_live.loc[test_mask_live, "Relevance"].copy()
qid_test_live = pdf_live.loc[test_mask_live, "qid"].copy()

podium_test_live = pdf_live.loc[test_mask_live, "PodiumFinish"].copy()

live_model_params = dict(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    objective="rank:ndcg",
    eval_metric="ndcg",
    tree_method="hist",
    early_stopping_rounds=30,
)

with mlflow.start_run(run_name="live_model_ranker") as run:
    mlflow.log_params(live_model_params)

    live_model = XGBRanker(**live_model_params)
    live_model.fit(
        X_train, y_train, qid=qid_train,
        eval_set=[(X_val_live, y_val_live)],
        eval_qid=[qid_val_live],
        verbose=False,
    )

    live_scores_test = live_model.predict(X_test)
    live_auc = roc_auc_score(podium_test_live, live_scores_test)

    eval_live_test = pdf_live.loc[test_mask_live, ["Year", "Circuit", "LapNumber", "PodiumFinish", "Relevance"]].copy()
    eval_live_test["score"] = live_scores_test
    live_top3 = topk_precision(eval_live_test, ["Year", "Circuit", "LapNumber"], "score", "PodiumFinish", k=3)

    # Plan F: same formalized-metrics-plus-bootstrap-CI treatment as
    # pre_model. The "race" unit here is one lap-snapshot, so there are far
    # more groups than pre_model's per-race count — expect a tighter CI,
    # which is itself informative about how much of live_model's apparent
    # accuracy comes from many correlated snapshots of the same race rather
    # than genuinely independent evidence.
    live_group_cols = ["Year", "Circuit", "LapNumber"]
    live_race_topk = eval_live_test.groupby(live_group_cols).apply(
        lambda g: topk_precision_group(g, "score", "PodiumFinish", k=3)
    )
    live_race_ndcg = eval_live_test.groupby(live_group_cols).apply(
        lambda g: ndcg_at_k_group(g, "score", "Relevance", k=3)
    )
    live_race_tau = eval_live_test.groupby(live_group_cols).apply(
        lambda g: kendall_tau_group(g, "score", "Relevance")
    )

    live_ndcg3 = live_race_ndcg.mean()
    live_tau = live_race_tau.mean()
    live_top3_ci = bootstrap_ci(live_race_topk)
    live_ndcg3_ci = bootstrap_ci(live_race_ndcg)
    live_tau_ci = bootstrap_ci(live_race_tau)

    mlflow.log_metric("test_auc_vs_podium", live_auc)
    mlflow.log_metric("test_top3_precision", live_top3)
    mlflow.log_metric("test_ndcg_at_3", live_ndcg3)
    mlflow.log_metric("test_kendall_tau", live_tau)
    mlflow.log_metric("test_top3_precision_ci_lo", live_top3_ci[0])
    mlflow.log_metric("test_top3_precision_ci_hi", live_top3_ci[2])
    mlflow.log_metric("test_ndcg_at_3_ci_lo", live_ndcg3_ci[0])
    mlflow.log_metric("test_ndcg_at_3_ci_hi", live_ndcg3_ci[2])
    mlflow.log_metric("test_kendall_tau_ci_lo", live_tau_ci[0])
    mlflow.log_metric("test_kendall_tau_ci_hi", live_tau_ci[2])
    mlflow.log_metric("test_n_snapshots", int(eval_live_test.groupby(live_group_cols).ngroups))

    live_model_signature = infer_signature(X_train, live_model.predict(X_train))
    mlflow.xgboost.log_model(
        live_model,
        "live_model",
        signature=live_model_signature,
        registered_model_name="workspace.default.f1_live_model",
    )

print("Live model AUC vs PodiumFinish:", round(live_auc, 4))
print(f"Live model top-3 precision: {live_top3:.4f}  (95% CI {live_top3_ci[0]:.4f}-{live_top3_ci[2]:.4f})")
print(f"Live model NDCG@3:          {live_ndcg3:.4f}  (95% CI {live_ndcg3_ci[0]:.4f}-{live_ndcg3_ci[2]:.4f})")
print(f"Live model Kendall's tau:   {live_tau:.4f}  (95% CI {live_tau_ci[0]:.4f}-{live_tau_ci[2]:.4f})")
print("Test lap-snapshots:", eval_live_test.groupby(live_group_cols).ngroups)

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(pre_model, max_num_features=15)
plt.show()

In [0]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(live_model, max_num_features=15)
plt.show()

In [ ]:
pre_scores_all = pre_model.predict(X_pre)
roc_auc_score(pdf_prerace["PodiumFinish"], pre_scores_all)

### Calibration — reinstated correctly (Plan F)

The old calibration-curve cell was removed because it compared `predict_proba` output against observed frequency — but `XGBRanker` has no `predict_proba`; `.predict()` returns a raw relevance score, only meaningful relative to other rows in the same `qid` group, not a calibrated per-row probability.

Plan F reinstates calibration the correct way for a ranker: fit a monotonic mapping from raw score to P(podium) on data the model didn't train on, then check that mapping's honesty on the test set.

- **Isotonic regression, not Platt scaling.** The relationship between ranker score and actual podium frequency isn't guaranteed to be sigmoid-shaped; isotonic regression only assumes it's monotonic (a higher score never implies a lower probability), which is the safer assumption here. The tradeoff is it needs more calibration data to avoid overfitting than Platt scaling does — with a full held-out validation season (2021) available for both models, that's an acceptable trade.
- **Fit on the validation split, never on train or test.** Fitting on train would calibrate against scores the model already memorized during boosting; evaluating on the same data used to fit the calibrator would hide overfitting in the reliability curve itself.
- **Brier score and log-loss are the headline numbers here, not AUC.** AUC only rewards relative ordering — which `rank:ndcg` already optimizes for directly — while Brier score and log-loss are what actually punish an overconfident or underconfident probability, which is the property a fan-facing "62% chance" number needs to be honest.

`predict_race_probabilities` further down (softmax over one race's raw scores) is a separate concern — a way to turn a *field* of scores into a distribution that sums to 100%. This calibration is about whether one driver's individual number is trustworthy in isolation.

In [ ]:
pre_scores_val = pre_model.predict(X_pre_val)
podium_pre_val = pdf_prerace.loc[val_mask, "PodiumFinish"].to_numpy()

pre_calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
pre_calibrator.fit(pre_scores_val, podium_pre_val)

pre_probs_test = pre_calibrator.predict(pre_scores_test)

pre_brier = brier_score_loss(podium_pre_test, pre_probs_test)
pre_logloss = log_loss(podium_pre_test, pre_probs_test, labels=[0, 1])

print("Pre-race calibrated Brier score (test, lower is better):", round(pre_brier, 4))
print("Pre-race calibrated log-loss   (test, lower is better):", round(pre_logloss, 4))

pre_frac_pos, pre_mean_pred = calibration_curve(podium_pre_test, pre_probs_test, n_bins=10, strategy="quantile")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(pre_mean_pred, pre_frac_pos, marker="o", label="pre_model (calibrated)")
ax.set_xlabel("Predicted P(podium)")
ax.set_ylabel("Observed podium frequency")
ax.set_title("Pre-race model calibration (test seasons)")
ax.legend()
plt.show()

In [ ]:
live_scores_val = live_model.predict(X_val_live)
podium_live_val = pdf_live.loc[val_mask_live, "PodiumFinish"].to_numpy()

live_calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
live_calibrator.fit(live_scores_val, podium_live_val)

live_probs_test = live_calibrator.predict(live_scores_test)

live_brier = brier_score_loss(podium_test_live, live_probs_test)
live_logloss = log_loss(podium_test_live, live_probs_test, labels=[0, 1])

print("Live model calibrated Brier score (test, lower is better):", round(live_brier, 4))
print("Live model calibrated log-loss   (test, lower is better):", round(live_logloss, 4))

live_frac_pos, live_mean_pred = calibration_curve(podium_test_live, live_probs_test, n_bins=10, strategy="quantile")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
ax.plot(live_mean_pred, live_frac_pos, marker="o", color="tab:orange", label="live_model (calibrated)")
ax.set_xlabel("Predicted P(podium)")
ax.set_ylabel("Observed podium frequency")
ax.set_title("Live model calibration (test seasons)")
ax.legend()
plt.show()

### Walk-forward (rolling-origin) validation — Plan F

The production `pre_model`/`live_model` above are each checked on a single static split (train ≤2020, val 2021, test ≥2022) — a real generalization check, but only against one train/test boundary. Walk-forward validation rolls that boundary forward one season at a time — train on everything before year *t*, test on year *t*, repeat for every *t* after an initial 3-season minimum — which is the standard shape for time-ordered data. It's a stronger claim than a single split: "this generalizes to the next season" checked three separate times (2021, 2022, and 2023 each held out in turn) instead of once.

This section trains fresh, throwaway models per fold purely for evaluation — it does **not** replace the single production model registered to Unity Catalog above. `phase_4.ipynb` and the Databricks App still load that one registered model, not a fold-specific one.

In [ ]:
def walk_forward_eval(pdf, feature_cols, group_cols, model_params, min_train_years=3):
    """
    Rolling-origin evaluation: for each test year after an initial minimum
    training window, train a fresh XGBRanker on every earlier year only
    (holding out the single most recent training year as the eval_set for
    early stopping — same convention as the production models above),
    evaluate on the held-out year, then move the origin forward one season.

    Returns (per-fold metrics DataFrame, {metric: (lo, mean, hi) bootstrap CI}
    pooled across every fold's races).
    """
    years = sorted(pdf["Year"].unique())
    fold_rows = []
    pooled_topk, pooled_ndcg, pooled_tau = [], [], []

    for i in range(min_train_years, len(years)):
        test_year = years[i]
        train_years = years[:i]
        if len(train_years) < 2:
            continue
        fold_val_year = train_years[-1]
        fold_train_years = train_years[:-1]
        if not fold_train_years:
            continue

        fold_train = pdf[pdf["Year"].isin(fold_train_years)].sort_values(group_cols).reset_index(drop=True)
        fold_val = pdf[pdf["Year"] == fold_val_year].sort_values(group_cols).reset_index(drop=True)
        fold_test = pdf[pdf["Year"] == test_year].sort_values(group_cols).reset_index(drop=True)

        if fold_train.empty or fold_val.empty or fold_test.empty:
            continue

        fold_qid = fold_train.groupby(group_cols, sort=True).ngroup()
        val_qid = fold_val.groupby(group_cols, sort=True).ngroup()

        fold_model = XGBRanker(**model_params)
        fold_model.fit(
            fold_train[feature_cols], fold_train["Relevance"], qid=fold_qid,
            eval_set=[(fold_val[feature_cols], fold_val["Relevance"])],
            eval_qid=[val_qid],
            verbose=False,
        )

        scores = fold_model.predict(fold_test[feature_cols])
        eval_df = fold_test[group_cols + ["PodiumFinish", "Relevance"]].copy()
        eval_df["score"] = scores

        race_topk = eval_df.groupby(group_cols).apply(lambda g: topk_precision_group(g, "score", "PodiumFinish", k=3))
        race_ndcg = eval_df.groupby(group_cols).apply(lambda g: ndcg_at_k_group(g, "score", "Relevance", k=3))
        race_tau = eval_df.groupby(group_cols).apply(lambda g: kendall_tau_group(g, "score", "Relevance"))

        fold_rows.append({
            "test_year": test_year,
            "train_years": f"{fold_train_years[0]}-{fold_train_years[-1]}",
            "n_groups": eval_df.groupby(group_cols).ngroups,
            "auc": roc_auc_score(eval_df["PodiumFinish"], eval_df["score"]),
            "top3_precision": race_topk.mean(),
            "ndcg@3": race_ndcg.mean(),
            "kendall_tau": race_tau.mean(),
        })
        pooled_topk.extend(race_topk.dropna().tolist())
        pooled_ndcg.extend(race_ndcg.dropna().tolist())
        pooled_tau.extend(race_tau.dropna().tolist())

    fold_df = pd.DataFrame(fold_rows)
    ci_summary = {
        "top3_precision": bootstrap_ci(pooled_topk),
        "ndcg@3": bootstrap_ci(pooled_ndcg),
        "kendall_tau": bootstrap_ci(pooled_tau),
    }
    return fold_df, ci_summary


pre_wf_folds, pre_wf_ci = walk_forward_eval(
    pdf_prerace, FEATURES_PRERACE, ["Year", "Circuit"], pre_model_params, min_train_years=3,
)
print("Pre-race walk-forward folds:")
display(pre_wf_folds)
for metric, (lo, mean, hi) in pre_wf_ci.items():
    print(f"  pooled {metric}: {mean:.4f}  (95% CI {lo:.4f}-{hi:.4f})")

In [ ]:
# live_model's walk-forward folds are heavier (lap-level rows, up to 600
# estimators with early stopping, per fold) than pre_model's — expect this
# cell to take noticeably longer. Reduce min_train_years or n_estimators
# here if iterating quickly; the params below match the production
# live_model exactly so the comparison against its static-split numbers is
# apples-to-apples.
live_wf_folds, live_wf_ci = walk_forward_eval(
    pdf_live, FEATURES_LIVE, ["Year", "Circuit", "LapNumber"], live_model_params, min_train_years=3,
)
print("Live model walk-forward folds:")
display(live_wf_folds)
for metric, (lo, mean, hi) in live_wf_ci.items():
    print(f"  pooled {metric}: {mean:.4f}  (95% CI {lo:.4f}-{hi:.4f})")

### What Plan F does *not* include yet

- **Bookmaker-odds benchmark.** The strongest possible KPI for a race-winner model — comparing Brier score / log-loss against historical bookmaker implied probabilities — needs a historical odds dataset this pipeline doesn't have. FastF1 and Ergast don't provide betting odds; adding that benchmark means picking and integrating a new external data source (a paid odds API, or a manually sourced historical-odds CSV), which is a data-acquisition decision for you to make deliberately, not something to fetch quietly as part of a metrics pass. Left as a follow-up.
- **Shadow-mode QA for live feature reconstruction.** This only makes sense once there's a live pipeline to shadow-test against — Plan D (the live-timing/Kafka companion) isn't built. `phase_4.ipynb`'s inference path still runs against the same batch-computed `f1_cleaned_lap_dataset` columns the offline model trained on, so there's no separate "live-reconstructed" feature path yet to diff against the offline gold values. Revisit this specifically if/when Plan D is implemented.

In [0]:
# Save Model to local filesystem only (serverless clusters cannot write to /dbfs/FileStore)
pre_model.save_model("../models/pre_model.json")
live_model.save_model("../models/live_model.json")

# Note: Copying to /dbfs/FileStore/models/ is not supported on serverless clusters. Models are saved in /models and can be downloaded from there if needed.

In [ ]:
### Trial

# Convert profile tables to pandas lookup tables for ad hoc inference. These
# are now (Driver, Year) / (TeamName, Year) grained (see the leakage fix
# above), so a lookup needs to pick a specific season, not just a driver.
driver_profile_pdf = driver_profile.toPandas()
constructor_profile_pdf = constructor_profile.toPandas()

DRIVER_PROFILE_FEATURES = [
    "driver_pace",
    "driver_sector1_skill",
    "driver_sector2_skill",
    "driver_sector3_skill",
    "driver_overtake_skill",
    "driver_speedI1",
    "driver_speedI2",
    "driver_speedFL",
    "driver_speedST",
    "lap_consistency",
    "tyre_management",
    "driver_avg_finish_score",
    "driver_quali_vs_race_delta",
    "driver_finish_consistency",
    "driver_win_rate",
    "driver_podium_rate",
    "driver_wet_finish_score",
    "driver_dry_finish_score",
    "driver_overtake_ability",
    "driver_dnf_rate",
    "driver_teammate_finish_delta",
    "driver_teammate_quali_delta",
    "driver_teammate_gain_delta",
]

CONSTRUCTOR_PROFILE_FEATURES = [
    "constructor_avg_finish_score",
    "constructor_avg_quali_score",
    "constructor_win_rate",
    "constructor_podium_rate",
    "constructor_dnf_rate",
    "constructor_season_strength",
]


def _latest_profile(pdf, key_col, key_value, year, feature_cols):
    # Most recent season at or before `year` as a proxy for current form —
    # never a season after the race being predicted, and never a blend
    # across seasons the way the pre-fix version accidentally was.
    candidates = pdf[(pdf[key_col] == key_value) & (pdf["Year"] <= year)]
    if candidates.empty:
        return pdf[feature_cols].mean(numeric_only=True).to_dict()
    latest = candidates.sort_values("Year").iloc[-1]
    return latest[feature_cols].to_dict()


def predict_race_probabilities(entries, circuit, year):
    """
    entries: list of {"driver": ..., "team": ..., "quali_pos": ...} dicts,
    one per car on the grid for a single race.

    Returns {driver: probability}, normalized to sum to 1 across `entries`.

    XGBRanker has no predict_proba, so there's no calibrated per-driver
    probability the way the old binary classifier had. Softmax over one
    race's raw ranker scores is the standard way to turn a set of ranking
    scores into a proper probability distribution — and it's arguably more
    honest than the old approach anyway, since independently-predicted
    per-driver percentages never summed to 100% across the field the way a
    single race's outcome actually must.
    """
    rows = []
    for e in entries:
        driver_values = _latest_profile(driver_profile_pdf, "Driver", e["driver"], year, DRIVER_PROFILE_FEATURES)
        constructor_values = _latest_profile(constructor_profile_pdf, "TeamName", e["team"], year, CONSTRUCTOR_PROFILE_FEATURES)

        row = {
            "Driver": e["driver"],
            "TeamName": e["team"],
            "Circuit": circuit,
            "Year": year,
            "QualiPosition": e["quali_pos"],
        }
        row.update(driver_values)
        row.update(constructor_values)
        rows.append(row)

    field = pd.DataFrame(rows)
    for c in CATEGORICAL_PRERACE:
        field[c] = field[c].astype("category")

    scores = pre_model.predict(field[FEATURES_PRERACE])
    exp_scores = np.exp(scores - scores.max())
    probs = exp_scores / exp_scores.sum()

    return dict(zip([e["driver"] for e in entries], probs))


In [0]:
l = list(pdf_prerace["TeamName"].unique())
print(l)

In [0]:
l = list(pdf_prerace["Circuit"].unique())
print(l)

In [ ]:
# Score every driver we've seen, all else equal (same hypothetical team and
# circuit, quali grid position assigned by list order) — softmax over the
# whole demo field, so the printed numbers actually sum to 100%.
demo_drivers = list(pdf_prerace["Driver"].unique())
demo_entries = [
    {"driver": d, "team": "Mercedes", "quali_pos": i + 1}
    for i, d in enumerate(demo_drivers)
]

demo_probs = predict_race_probabilities(demo_entries, circuit="Silverstone", year=2023)

for driver, prob in sorted(demo_probs.items(), key=lambda kv: -kv[1])[:10]:
    print(f"{driver} - {prob * 100:.2f}%")